# ♟️ Chess-GPT — Experiment Notebook

**Goal:** Train a small GPT to predict chess moves in PGN notation using a move-level tokenizer.

```
Input game:   1. e4 e5 2. Nf3 Nc6 3. Bb5 a6 ...
Model output: next move in the sequence
```

**Roadmap:**
1. ✅ Imports & sanity checks
2. ✅ Dataset choice & download
3. ✅ PGN parsing → corpus
4. ✅ Game length analysis
5. ✅ Build move-level tokenizer (1 token = 1 SAN move)
6. ✅ Build PyTorch Dataset → DataLoader
7. ✅ Train small GPT
8. ✅ Generate & validate moves with `python-chess`
9. ✅ Constrained decoding — mask illegal moves to `-inf` before softmax

---
## 0 · Setup — imports & paths

> Run from `ml-playground/chess-gpt/`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

from pathlib import Path
import torch
import re

DEVICE = (
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Device: {DEVICE}')
Path('artifacts').mkdir(exist_ok=True)
Path('data').mkdir(exist_ok=True)

# ── Loguru + TensorBoard ──────────────────────────────────────────────────────
from src.logging_utils import setup_loguru, make_tb_writer

RUN_NAME = 'chess_gpt_move'
setup_loguru(log_dir='artifacts/logs', run_name=RUN_NAME)
writer = make_tb_writer(log_dir='artifacts/runs', run_name=RUN_NAME)
print(f'TensorBoard: tensorboard --logdir artifacts/runs')

---
## 1 · Dataset choice

### Why `lichess_db_standard_rated_2014-07.pgn.zst`?

| Dataset | Size | Games | Why / Why not |
|---|---|---|---|
| **Lichess monthly (2014-07)** | ~50 MB uncompressed | ~500K | ✅ Small, fast, free, rated games |
| Lichess monthly (2024) | ~50 GB | ~100M | ❌ Too large for first run |
| KingBase (~2021) | ~2 GB | ~2.4M | ⚠️  Only elite games, less diverse |
| CCRL (computer chess) | ~1 GB | ~1.5M | ⚠️  Computer play, unusual patterns |
| chess.com games API | varies | varies | ⚠️  Requires API key |

**Pick:** Lichess July 2014 — small enough to iterate in minutes on a laptop.

```bash
# Install zstd decompressor once:
brew install zstd   # macOS
# apt install zstd  # Linux
```

In [ ]:
# ── Download Lichess July 2014 (≈ 22 MB compressed) ─────────────────────────
#
# If you already have a PGN file somewhere, skip this cell and set PGN_PATH below.
#
import urllib.request, subprocess

URL      = 'https://database.lichess.org/standard/lichess_db_standard_rated_2014-07.pgn.zst'
ZST_PATH = Path('data/lichess_2014_07.pgn.zst')
PGN_PATH = Path('data/lichess_2014_07.pgn')

if not PGN_PATH.exists():
    if not ZST_PATH.exists():
        print('Downloading …')
        urllib.request.urlretrieve(URL, ZST_PATH)
        print(f'Downloaded → {ZST_PATH}')
    print('Decompressing …')
    subprocess.run(['zstd', '-d', str(ZST_PATH), '-o', str(PGN_PATH)], check=True)
    print(f'Decompressed → {PGN_PATH}')
else:
    print(f'PGN already exists: {PGN_PATH}  ({PGN_PATH.stat().st_size / 1e6:.1f} MB)')

---
## 2 · Parse PGN → corpus

We **strip all metadata headers** and keep only the move sequences.  
Each game becomes one line:
```
1. e4 e5 2. Nf3 Nc6 3. Bb5 a6 4. Ba4 Nf6 ...
```

In [ ]:
def parse_pgn_moves(pgn_path: str, max_games: int = 50_000) -> list[str]:
    """
    Extract move-only strings from a PGN file.

    Strips:
      - Header lines ([Event ...], [White ...], etc.)
      - Clock annotations ({ [%clk 0:05:00] })
      - Game-result tokens (1-0, 0-1, 1/2-1/2, *)

    Returns a list of strings like:
        ['1. e4 e5 2. Nf3 Nc6 ...', ...]
    """
    games = []
    current_moves = []
    in_moves = False

    result_re = re.compile(r'(1-0|0-1|1/2-1/2|\*)\s*$')
    clock_re  = re.compile(r'\{[^}]*\}')          # { ... } annotations
    eol_re    = re.compile(r'\s+')

    with open(pgn_path, encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()

            if line.startswith('['):               # header
                if in_moves and current_moves:
                    games.append(' '.join(current_moves))
                    current_moves = []
                    if len(games) >= max_games:
                        break
                in_moves = False
                continue

            if not line:                           # blank line between header & moves
                in_moves = True
                continue

            if in_moves:
                line = clock_re.sub('', line)      # strip { clk }
                line = result_re.sub('', line)     # strip 1-0 etc
                line = eol_re.sub(' ', line).strip()
                if line:
                    current_moves.append(line)

    # Flush last game
    if current_moves and len(games) < max_games:
        games.append(' '.join(current_moves))

    return games


# ── Load ────────────────────────────────────────────────────────────────────
MAX_GAMES = 30_000   # ← tweak: 5K for quick test, 50K for a real run

games = parse_pgn_moves(str(PGN_PATH), max_games=MAX_GAMES)
print(f'Parsed {len(games):,} games')
print('Sample game (first 200 chars):')
print(games[0][:200])

In [ ]:
# Quick stats
lengths = [len(g.split()) for g in games]
print(f'Avg moves/game : {sum(lengths)/len(lengths):.1f}')
print(f'Max moves/game : {max(lengths)}')
print(f'Min moves/game : {min(lengths)}')

corpus = '\n'.join(games)   # one game per line
print(f'\nCorpus size    : {len(corpus):,} chars')

In [ ]:
---
## 2b · Game Length Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter

# ── Compute per-game move counts (half-moves = plies) ───────────────────────
lengths   = [len(g.split()) for g in games]          # tokens including move numbers
# Count only actual moves (strip move numbers: tokens ending with '.')
ply_counts = []
for g in games:
    toks = g.split()
    plies = [t for t in toks if not t.endswith('.')]
    ply_counts.append(len(plies))

lengths_arr = np.array(ply_counts)

print('── Game Length Statistics (half-moves / plies) ──')
print(f'  Count   : {len(lengths_arr):,}')
print(f'  Mean    : {lengths_arr.mean():.1f}')
print(f'  Median  : {np.median(lengths_arr):.1f}')
print(f'  Std     : {lengths_arr.std():.1f}')
print(f'  Min     : {lengths_arr.min()}')
print(f'  Max     : {lengths_arr.max()}')
print()
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f'  P{p:<2}     : {np.percentile(lengths_arr, p):.0f}')

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── 1. Histogram + KDE ───────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.hist(lengths_arr, bins=80, color='steelblue', alpha=0.7, edgecolor='white', linewidth=0.4, density=True, label='Histogram')
from scipy.stats import gaussian_kde
kde = gaussian_kde(lengths_arr, bw_method=0.1)
xs  = np.linspace(lengths_arr.min(), lengths_arr.max(), 400)
ax1.plot(xs, kde(xs), color='tomato', lw=2, label='KDE')
for p, col in [(50, 'gold'), (90, 'orange'), (99, 'red')]:
    v = np.percentile(lengths_arr, p)
    ax1.axvline(v, color=col, ls='--', lw=1.2, label=f'P{p}={v:.0f}')
ax1.set_title('Game Length Distribution (plies)', fontsize=13)
ax1.set_xlabel('Half-moves (plies)')
ax1.set_ylabel('Density')
ax1.legend(fontsize=9)

# ── 2. Box plot ──────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
bp = ax2.boxplot(lengths_arr, vert=True, patch_artist=True,
                 boxprops=dict(facecolor='steelblue', alpha=0.6),
                 medianprops=dict(color='tomato', lw=2),
                 flierprops=dict(marker='.', color='gray', alpha=0.3, markersize=3))
ax2.set_title('Box Plot', fontsize=13)
ax2.set_ylabel('Plies')
ax2.set_xticks([])

# ── 3. Cumulative distribution ───────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, :2])
sorted_len = np.sort(lengths_arr)
cdf = np.arange(1, len(sorted_len) + 1) / len(sorted_len)
ax3.plot(sorted_len, cdf * 100, color='steelblue', lw=2)
for p, col in [(50, 'gold'), (75, 'orange'), (90, 'red'), (95, 'darkred')]:
    v = np.percentile(lengths_arr, p)
    ax3.axvline(v, color=col, ls='--', lw=1.2, label=f'P{p}={v:.0f}')
    ax3.axhline(p, color=col, ls=':', lw=0.8, alpha=0.5)
ax3.set_title('Cumulative Distribution of Game Lengths', fontsize=13)
ax3.set_xlabel('Plies')
ax3.set_ylabel('% of games ≤ length')
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

# ── 4. Game length buckets bar chart ─────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
buckets = {'≤20': 0, '21-40': 0, '41-60': 0, '61-80': 0, '81-100': 0, '>100': 0}
for l in lengths_arr:
    if   l <= 20:  buckets['≤20']    += 1
    elif l <= 40:  buckets['21-40']  += 1
    elif l <= 60:  buckets['41-60']  += 1
    elif l <= 80:  buckets['61-80']  += 1
    elif l <= 100: buckets['81-100'] += 1
    else:          buckets['>100']   += 1
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6']
bars = ax4.bar(list(buckets.keys()), [v / len(lengths_arr) * 100 for v in buckets.values()],
               color=colors, edgecolor='white', linewidth=0.5)
for bar, (k, v) in zip(bars, buckets.items()):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{v/len(lengths_arr)*100:.1f}%', ha='center', va='bottom', fontsize=8)
ax4.set_title('Game Length Buckets', fontsize=13)
ax4.set_ylabel('% of games')
ax4.tick_params(axis='x', rotation=30)

fig.suptitle(f'Chess Game Length Analysis  (n={len(lengths_arr):,} games)', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

print(f'\nGames ≤ 40 plies  (very short) : {buckets["≤20"] + buckets["21-40"]:,}  ({(buckets["≤20"]+buckets["21-40"])/len(lengths_arr)*100:.1f}%)')
print(f'Games 41-80 plies (typical)    : {buckets["41-60"] + buckets["61-80"]:,}  ({(buckets["41-60"]+buckets["61-80"])/len(lengths_arr)*100:.1f}%)')
print(f'Games > 80 plies  (long)       : {buckets["81-100"] + buckets[">100"]:,}  ({(buckets["81-100"]+buckets[">100"])/len(lengths_arr)*100:.1f}%)')

---
## 3b · Move-Level Tokenizer

**Why move-level instead of BPE for constrained decoding?**

With BPE, a single token might encode *half a move* or *multiple moves*, making it impossible to directly map token IDs to legal SAN strings.

With a **move-level tokenizer** each token = exactly one SAN move (`e4`, `Nf3`, `O-O`, `Bb5+`).
At generation time we can do:

```python
legal_ids = [tok.move_to_id[board.san(m)] for m in board.legal_moves]
logits[~legal_ids] = -inf    # ← trivial mask
```

In [ ]:
from src.tokenizers import MoveTokenizer

MOV_TOK_PATH = Path('artifacts/move_tok.pkl')

if MOV_TOK_PATH.exists():
    print('Loading existing MoveTokenizer…')
    mtok = MoveTokenizer.load(MOV_TOK_PATH)
else:
    print('Building MoveTokenizer…')
    mtok = MoveTokenizer().build(games)
    mtok.save(MOV_TOK_PATH)

print(f'Vocab size : {mtok.vocab_size}')
print(f'PAD={mtok.pad_id}  BOS={mtok.bos_id}  EOS={mtok.eos_id}  UNK={mtok.unk_id}')

# ── Sanity: encode → decode ──────────────────────────────────────────────────
sample_game = games[0]
ids_m  = mtok.encode(sample_game, add_special=True)
back_m = mtok.decode(ids_m)
print(f'\nSample (first 120 chars) : {sample_game[:120]}')
print(f'Encoded ({len(ids_m)} tokens)      : {ids_m[:12]} …')
print(f'Decoded                  : {back_m[:120]}')

In [ ]:
import chess
import chess.pgn

def validate_pgn(game_str: str) -> dict:
    """Count how many consecutive moves are legal."""
    board = chess.Board()
    tokens = game_str.split()
    legal, total = 0, 0
    for tok_str in tokens:
        if tok_str.endswith('.') or tok_str in ('1-0', '0-1', '1/2-1/2', '*'):
            continue
        try:
            move = board.parse_san(tok_str)
            board.push(move)
            legal += 1
        except (chess.InvalidMoveError, chess.IllegalMoveError, chess.AmbiguousMoveError, ValueError):
            total += 1
            break
        total += 1
    return {'legal': legal, 'total': total, 'legal_pct': legal / max(total, 1) * 100}

print('validate_pgn() ready.')

---
## Next steps / experiments

```
[ ] Scale up: 30K → 200K games
[ ] Bigger model: d_model=256, n_layers=6
[ ] Conditional prompting: Elo-conditioned generation
[ ] Nucleus (top-p) sampling for better move diversity
[ ] Filter dataset: rated 1800+ Elo for higher-quality training games
[ ] Beam search over legal moves only (constrained beam)
```

---
## 8 · Train GPT with MoveTokenizer

Re-use the same GPT architecture but swap the tokenizer.  
Shorter sequences (1 token = 1 move) → faster training, and we unlock **constrained decoding**.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MoveLevelDataset(Dataset):
    """
    Same windowed approach as ChessDataset but uses MoveTokenizer.
    Each sequence window contains exactly `max_len` move tokens.
    """

    def __init__(self, games: list[str], tokenizer: MoveTokenizer, max_len: int = 128):
        all_ids = tokenizer.encode_batch(games, add_special=True)
        flat = []
        for ids in all_ids:
            flat.extend(ids)
        self.tokens  = flat
        self.max_len = max_len
        self.n       = (len(flat) - 1) // max_len
        print(f'MoveLevelDataset | total_tokens={len(flat):,} | windows={self.n:,}')

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        start = idx * self.max_len
        end   = start + self.max_len
        return {
            'input_ids': torch.tensor(self.tokens[start:end],     dtype=torch.long),
            'labels':    torch.tensor(self.tokens[start+1:end+1], dtype=torch.long),
        }


MOVE_SEQ_LEN  = 128    # shorter context window (1 token = 1 move)
MOVE_BATCH_SZ = 64

m_train_ds = MoveLevelDataset(train_games, mtok, max_len=MOVE_SEQ_LEN)
m_val_ds   = MoveLevelDataset(val_games,   mtok, max_len=MOVE_SEQ_LEN)

m_train_loader = DataLoader(m_train_ds, batch_size=MOVE_BATCH_SZ, shuffle=True)
m_val_loader   = DataLoader(m_val_ds,   batch_size=MOVE_BATCH_SZ, shuffle=False)

print(f'Train batches : {len(m_train_loader):,}')
print(f'Val   batches : {len(m_val_loader):,}')

In [ ]:
from src.model import GPT2, GPT2Config

move_config = GPT2Config(
    vocab_size  = mtok.vocab_size,
    pad_id      = mtok.pad_id,
    d_model     = 128,
    n_heads     = 4,
    n_layers    = 4,
    d_ff        = 512,
    max_seq_len = MOVE_SEQ_LEN,
    dropout     = 0.1,
    batch_size  = MOVE_BATCH_SZ,
    lr          = 3e-4,
    epochs      = 5,
)

move_model = GPT2(move_config).to(DEVICE)
print(f'Parameters : {move_model.n_params:,}')
print(f'Vocab size : {move_config.vocab_size}  (one ID per unique SAN move)')

In [ ]:
from torch.optim import AdamW
import math
from src.metrics import top_k_accuracy, move_rank as move_rank_metric
from src.logging_utils import log_epoch_metrics
from loguru import logger

m_optimizer = AdamW(move_model.parameters(), lr=move_config.lr, weight_decay=0.01)
m_criterion = torch.nn.CrossEntropyLoss(ignore_index=move_config.pad_id)

history = []
global_step = 0


def run_epoch_move(model, loader, optimizer=None, train=True):
    global global_step
    model.train() if train else model.eval()
    total_loss = total_top1 = total_top5 = total_rank = 0.0
    n = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            ids    = batch['input_ids'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            logits = model(ids)
            loss   = m_criterion(logits.reshape(-1, move_config.vocab_size), labels.reshape(-1))
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                writer.add_scalar('step/train_loss', loss.item(), global_step)
                global_step += 1
            total_loss += loss.item()
            with torch.no_grad():
                total_top1 += top_k_accuracy(logits, labels, move_config.pad_id, k=1)
                total_top5 += top_k_accuracy(logits, labels, move_config.pad_id, k=5)
                r = move_rank_metric(logits, labels, move_config.pad_id)
                total_rank += r if not math.isnan(r) else 0.0
            n += 1
    avg_loss = total_loss / n
    return {
        'loss': avg_loss,
        'ppl':  math.exp(min(avg_loss, 20)),
        'top1': total_top1 / n * 100,
        'top5': total_top5 / n * 100,
        'rank': total_rank / n,
    }


logger.info(f'Training start | epochs={move_config.epochs} lr={move_config.lr} '
            f'batch={move_config.batch_size} vocab={move_config.vocab_size}')

print(f'{"Ep":<4} {"TrLoss":<8} {"TrPPL":<8} {"TrTop1":<8} '
      f'{"VlLoss":<8} {"VlPPL":<8} {"VlTop1":<8} {"VlTop5":<8} {"VlRank":<7}')
print('─' * 67)

for epoch in range(1, move_config.epochs + 1):
    tr = run_epoch_move(move_model, m_train_loader, m_optimizer, train=True)
    vl = run_epoch_move(move_model, m_val_loader,   train=False)
    history.append({'epoch': epoch, 'train': tr, 'val': vl})
    log_epoch_metrics(writer, epoch, tr, vl)
    print(f'{epoch:<4} {tr["loss"]:<8.4f} {tr["ppl"]:<8.1f} {tr["top1"]:<8.1f} '
          f'{vl["loss"]:<8.4f} {vl["ppl"]:<8.1f} {vl["top1"]:<8.1f} '
          f'{vl["top5"]:<8.1f} {vl["rank"]:<7.1f}')

writer.add_hparams(
    {'d_model': move_config.d_model, 'n_layers': move_config.n_layers,
     'n_heads': move_config.n_heads, 'lr': move_config.lr},
    {'hparam/val_loss': history[-1]['val']['loss'],
     'hparam/val_top1': history[-1]['val']['top1']},
)
writer.flush()
torch.save(move_model.state_dict(), 'artifacts/chess_gpt_move.pt')
logger.success('Training complete → artifacts/chess_gpt_move.pt')

---
## 9 · Constrained Decoding

At each step we:
1. Get all legal moves from `chess.Board` in SAN notation
2. Look up their token IDs in `MoveTokenizer`
3. Set **all other logits to `-inf`** before softmax → model can only sample a legal move

```
logits[illegal_token_ids] = -inf
probs = softmax(logits)         # only legal moves have nonzero prob
```

No beam search, no external solver — just a one-line logit mask.

In [ ]:
import chess

@torch.no_grad()
def generate_constrained(
    prompt: str,
    model: GPT2,
    tokenizer: MoveTokenizer,
    max_new_moves: int = 60,
    temperature: float = 0.8,
    top_k: int = None,
) -> str:
    """
    Generate a chess game with constrained decoding.

    At every step:
        1. Get legal SAN moves from the board.
        2. Find their token IDs.
        3. Mask all other logits to -inf before softmax.
        4. Sample — the model can only pick a legal move.

    Args:
        prompt:        PGN-like string with starting moves, e.g. '1. e4 e5 2. Nf3'
        model:         Trained GPT (move-level tokenizer expected)
        tokenizer:     MoveTokenizer instance
        max_new_moves: Maximum moves to generate
        temperature:   Sampling temperature
        top_k:         If set, further restrict to top-k legal moves by logit value

    Returns:
        Full PGN-like string (prompt + generated continuation)
    """
    model.eval()
    board = chess.Board()

    # ── Apply prompt moves to board ──────────────────────────────────────────
    prompt_moves = tokenizer._split_moves(prompt)
    for san in prompt_moves:
        try:
            board.push(board.parse_san(san))
        except Exception:
            break   # stop at first invalid move in prompt

    # ── Encode prompt ────────────────────────────────────────────────────────
    ids    = tokenizer.encode(prompt, add_special=False)
    tensor = torch.tensor([ids], dtype=torch.long, device=next(model.parameters()).device)

    generated_moves: list[str] = []

    for _ in range(max_new_moves):
        if board.is_game_over():
            break

        # Legal moves in SAN → token IDs
        legal_sans = [board.san(m) for m in board.legal_moves]
        legal_ids  = [tokenizer.move_to_id[s] for s in legal_sans if s in tokenizer.move_to_id]

        if not legal_ids:
            break   # no known token for any legal move (very rare for unseen positions)

        # Forward pass — use last `max_seq_len` tokens as context
        ctx    = tensor[:, -model.config.max_seq_len:]
        logits = model(ctx)[:, -1, :] / temperature   # (1, V)

        # ── Mask: set all illegal tokens to -inf ─────────────────────────────
        mask = torch.full_like(logits, float('-inf'))
        mask[0, legal_ids] = logits[0, legal_ids]

        # Optional top-k within legal moves
        if top_k is not None and top_k < len(legal_ids):
            topk_vals, _ = torch.topk(mask, top_k)
            mask[mask < topk_vals[:, -1:]] = float('-inf')

        probs      = torch.softmax(mask, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)   # (1, 1)

        # ── Update board state ───────────────────────────────────────────────
        move_san = tokenizer.id_to_move.get(next_token.item(), '')
        try:
            board.push(board.parse_san(move_san))
        except Exception:
            break   # should never happen — but bail gracefully

        generated_moves.append(move_san)
        tensor = torch.cat([tensor, next_token], dim=1)

    # ── Decode full sequence ─────────────────────────────────────────────────
    return tokenizer.decode(tensor[0].tolist())


print('generate_constrained() ready.')

In [ ]:
# ── Constrained generation demo ─────────────────────────────────────────────
prompts = [
    '1. e4',
    '1. d4',
    '1. e4 e5 2. Nf3',
    '1. e4 c5',   # Sicilian
]

print('=' * 70)
print('CONSTRAINED DECODING  (all moves guaranteed legal)')
print('=' * 70)
for p in prompts:
    game = generate_constrained(p, move_model, mtok, max_new_moves=30, temperature=0.8)
    result = validate_pgn(game)
    print(f'\nPrompt : {p}')
    print(f'Output : {game[:200]}')
    print(f'Legal  : {result["legal"]}/{result["total"]}  ({result["legal_pct"]:.0f}%)')

In [ ]:
# ── Side-by-side: unconstrained vs constrained ───────────────────────────────
def generate_unconstrained(
    prompt: str,
    model: GPT2,
    tokenizer: MoveTokenizer,
    max_new_moves: int = 30,
    temperature: float = 0.8,
    top_k: int = 40,
) -> str:
    """Standard sampling — no legality enforcement."""
    model.eval()
    ids    = tokenizer.encode(prompt, add_special=False)
    tensor = torch.tensor([ids], dtype=torch.long, device=next(model.parameters()).device)
    with torch.no_grad():
        out = model.generate(tensor, max_new_tokens=max_new_moves, temperature=temperature, top_k=top_k)
    return tokenizer.decode(out[0].tolist())


PROMPT = '1. e4 e5 2. Nf3'
N_SAMPLES = 5

print(f'Prompt: {PROMPT}\n')
print(f'{"─"*34}  UNCONSTRAINED  {"─"*34}')
unc_legal = []
for i in range(N_SAMPLES):
    g = generate_unconstrained(PROMPT, move_model, mtok, max_new_moves=20)
    r = validate_pgn(g)
    unc_legal.append(r['legal_pct'])
    print(f'  [{i+1}] {g[:120]}')
    print(f'       legal={r["legal"]}/{r["total"]} ({r["legal_pct"]:.0f}%)')

print(f'\n{"─"*34}  CONSTRAINED  {"─"*36}')
con_legal = []
for i in range(N_SAMPLES):
    g = generate_constrained(PROMPT, move_model, mtok, max_new_moves=20)
    r = validate_pgn(g)
    con_legal.append(r['legal_pct'])
    print(f'  [{i+1}] {g[:120]}')
    print(f'       legal={r["legal"]}/{r["total"]} ({r["legal_pct"]:.0f}%)')

print(f'\n── Summary ──')
print(f'  Unconstrained avg legal% : {sum(unc_legal)/len(unc_legal):.1f}%')
print(f'  Constrained   avg legal% : {sum(con_legal)/len(con_legal):.1f}%')

In [ ]:
import matplotlib.pyplot as plt

epochs_x  = [h['epoch'] for h in history]
tr_loss   = [h['train']['loss']  for h in history]
vl_loss   = [h['val']['loss']    for h in history]
tr_ppl    = [h['train']['ppl']   for h in history]
vl_ppl    = [h['val']['ppl']     for h in history]
tr_top1   = [h['train']['top1']  for h in history]
vl_top1   = [h['val']['top1']    for h in history]
vl_top5   = [h['val']['top5']    for h in history]
vl_rank   = [h['val']['rank']    for h in history]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].plot(epochs_x, tr_loss, 'o-', label='train')
axes[0].plot(epochs_x, vl_loss, 's--', label='val')
axes[0].set_title('Cross-Entropy Loss')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, tr_ppl, 'o-', label='train')
axes[1].plot(epochs_x, vl_ppl, 's--', label='val')
axes[1].set_title('Perplexity')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_x, tr_top1, 'o-', label='top-1 train')
axes[2].plot(epochs_x, vl_top1, 's--', label='top-1 val')
axes[2].plot(epochs_x, vl_top5, '^:', label='top-5 val')
axes[2].set_title('Move Accuracy (%)')
axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(alpha=0.3)

axes[3].plot(epochs_x, vl_rank, 's--', color='tomato')
axes[3].set_title('Median Move Rank (val)\nlower = better')
axes[3].set_xlabel('Epoch'); axes[3].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 10 · Full Evaluation

All metrics in one place after training.

| Metric | Where computed | What it measures |
|---|---|---|
| **Cross-entropy loss** | training loop (batch) | How well model predicts next token |
| **Perplexity** | training loop (batch) | Exponentiated loss — lower = more confident |
| **Top-1 accuracy** | training loop (batch) | Model's top prediction = true next move |
| **Top-5 accuracy** | training loop (batch) | True move in model's top-5 predictions |
| **Median move rank** | training loop (batch) | Rank of true move in sorted predictions |
| **Legal move rate** | game-level (generated) | Mean % moves legal before first error |
| **Game completion rate** | game-level (generated) | % games with zero illegal moves |
| **Avg legal length** | game-level (generated) | Mean half-moves before first illegal move |

In [ ]:
from src.metrics import game_metrics
from src.logging_utils import log_game_metrics

EVAL_PROMPTS = ['1. e4', '1. d4', '1. e4 e5 2. Nf3', '1. e4 c5', '1. Nf3', '1. c4']
N_EVAL       = 20   # games per condition

logger.info(f'Game-level evaluation  |  {N_EVAL} games × {len(EVAL_PROMPTS)} prompts')

# ── Generate games ────────────────────────────────────────────────────────────
unc_games = [generate_unconstrained('1. e4', move_model, mtok, max_new_moves=40)
             for _ in range(N_EVAL)]
con_games = [generate_constrained  ('1. e4', move_model, mtok, max_new_moves=40)
             for _ in range(N_EVAL)]

# ── Compute game metrics ──────────────────────────────────────────────────────
unc_m = game_metrics(unc_games)
con_m = game_metrics(con_games)

# ── Log to TensorBoard + loguru ───────────────────────────────────────────────
last_epoch = history[-1]['epoch'] if history else 0
log_game_metrics(writer, unc_m, step=last_epoch, tag_prefix='eval/unconstrained')
log_game_metrics(writer, con_m, step=last_epoch, tag_prefix='eval/constrained')
writer.flush()

# ── Print summary table ───────────────────────────────────────────────────────
print(f'\n{"Metric":<26}  {"Unconstrained":>15}  {"Constrained":>13}')
print('─' * 58)
print(f'{"Legal move rate (%)":<26}  {unc_m["legal_move_rate"]:>15.1f}  {con_m["legal_move_rate"]:>13.1f}')
print(f'{"Game completion rate (%)":<26}  {unc_m["game_completion_rate"]:>15.1f}  {con_m["game_completion_rate"]:>13.1f}')
print(f'{"Avg legal length (plies)":<26}  {unc_m["avg_legal_length"]:>15.1f}  {con_m["avg_legal_length"]:>13.1f}')
print(f'{"Games evaluated":<26}  {unc_m["n_games"]:>15}  {con_m["n_games"]:>13}')
print(f'\nTensorBoard: tensorboard --logdir artifacts/runs')